In [25]:
import pandas as pd
import numpy as np
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from process_census_data import prepare_census_data, load_raw_data

In [26]:
train_x, test_x, train_y, test_y = prepare_census_data(
    random_state=99,
    split_validation=False,
    rebalance=True,
)

train_x.head()

,class of worker_ Federal government,class of worker_ Local government,class of worker_ Never worked,class of worker_ Not in universe,class of worker_ Private,class of worker_ Self-employed-incorporated,class of worker_ Self-employed-not incorporated,class of worker_ State government,class of worker_ Without pay,enroll in edu inst last wk_ College or university,...,num persons worked for employer,weeks worked in year,education-num,member of a labor union,live in this house 1 year ago,own business or self employed,fill inc questionnaire for veteran's admin,veterans benefits,year,sex
88942,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,-1,0,0,-1,0,1,0
121916,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,6.0,52.0,14.0,-1,-1,0,-1,-1,0,1
18744,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,-1,-1,0,-1,0,0,0
33098,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,1.0,52.0,2.0,-1,1,0,-1,-1,1,1
125779,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,5.0,-1,-1,0,-1,-1,0,1


In [27]:
my_model = RandomForestClassifier(
    n_estimators=500,
    random_state=99,
    min_samples_split=500,
    max_leaf_nodes=50,
).fit(train_x, train_y)
test_pred = my_model.predict(test_x)

print(classification_report(test_y, test_pred, zero_division=0))
print(confusion_matrix(test_y, test_pred))

              precision    recall  f1-score   support

           0       0.99      0.82      0.90     46753
           1       0.25      0.89      0.39      3128

    accuracy                           0.82     49881
   macro avg       0.62      0.86      0.64     49881
weighted avg       0.94      0.82      0.86     49881

[[38199  8554]
 [  331  2797]]


# Practical: Explainable AI

In this model, we currently use 126 input features to predict whether an individual earns more than $50K per year. While this rich feature set may yield strong predictive performance, it also makes the model more complex, less interpretable, and potentially more costly to maintain.
The goal of this exercise is to apply Explainable AI (XAI) techniques to identify the most informative features and reduce the model complexity.

### Task:
Use XAI methods to select a subset of only 5 features that still allows the model to achieve the highest possible predictive performance.
In particular, you should:
- Apply SHAP values and permutation feature importance to understand the contribution of each feature.
- Rank the features based on their importance and predictive contribution.
- Select the top 5 most relevant features.
- Retrain the model using only these features.
- Compare the performance of the reduced model with the original full model.

### Objective:
Find the best trade-off between model interpretability and predictive accuracy, demonstrating that a much smaller set of features can still deliver strong performance.

## Bonus Exercise: Detecting Bias in Sensitive Variables
In addition to model performance and interpretability, it is important to evaluate whether the model behaves fairly across different subgroups. In this dataset, race and sex are considered sensitive attributes. Your task is to investigate whether the model exhibits bias with respect to these variables.

### Task
Use Explainable AI techniques to assess whether the model treats individuals differently based on their race or sex.
Specifically:


### SHAP-based analysis
- Compute SHAP values for the model.
- Analyze how the features race and sex contribute to the predictions.
- Compare the distribution of SHAP values across different groups (e.g., male vs female).
- Identify whether belonging to a specific group systematically increases or decreases the predicted probability of earning >50K.

### Counterfactual analysis
- For a given individual, create counterfactual examples by changing only the sensitive attribute (e.g., switch sex or race).
- Keep all other features constant.
- Observe whether the prediction changes significantly.
- Repeat this for multiple individuals to assess whether such changes occur systematically.

### Questions to answer

- Do race or sex have a strong direct influence on the predictions?
- Are there systematic differences in predictions between groups?
- Do counterfactual changes in sensitive variables lead to meaningful changes in outcomes?
- Is the observed effect justified (proxy relationships, correlations) or does it indicate potential unfairness?

In [28]:
import eli5
from eli5.sklearn import PermutationImportance

model_features = list(getattr(my_model, "feature_names_in_", test_x.columns))
test_x_aligned = test_x.reindex(columns=model_features, fill_value=0)

perm = PermutationImportance(
    my_model,
    random_state=99,
    scoring="f1",
)
perm.fit(test_x_aligned, test_y)

perm_importance = (
    pd.Series(
        perm.feature_importances_,
        index=model_features,
        name="permutation_importance",
    )
    .sort_values(ascending=False)
 )

perm_summary = perm_importance.head(50).to_frame()
perm_summary

,permutation_importance
sex,0.019808
capital gains,0.006545
dividends from stocks,0.004737
education-num,0.004214
class of worker_ Private,0.002135
hispanic origin_ Mexican (Mexicano),0.002035
capital losses,0.001593
hispanic origin_ All other,0.001259
veterans benefits,0.000824
wage per hour,0.000432


In [29]:
analysis_sample = test_x_aligned.sample(min(1000, len(test_x_aligned)), random_state=99)

explainer = shap.TreeExplainer(my_model)
shap_values_sample = explainer.shap_values(analysis_sample)

if isinstance(shap_values_sample, list):
    local_shap_values = shap_values_sample[1] if len(shap_values_sample) > 1 else shap_values_sample[0]
elif isinstance(shap_values_sample, np.ndarray) and shap_values_sample.ndim == 3:
    local_shap_values = shap_values_sample[:, :, 1]
else:
    local_shap_values = shap_values_sample

shap_importance = (
    pd.Series(
        np.abs(local_shap_values).mean(axis=0),
        index=analysis_sample.columns,
        name="mean_abs_shap",
    )
    .sort_values(ascending=False)
 )

shap_summary = shap_importance.head(50).to_frame()
shap_summary

,mean_abs_shap
weeks worked in year,0.066761
education-num,0.065622
num persons worked for employer,0.043279
age,0.040505
sex,0.039790
tax filer stat_ Nonfiler,0.037836
dividends from stocks,0.033563
detailed household summary in household_ Householder,0.031447
class of worker_ Not in universe,0.026544
marital stat_ Never married,0.017187


In [30]:
summary_features = pd.concat(
    [perm_importance, shap_importance],
    axis=1,
    join="inner",
)

summary_features["permutation_rank"] = summary_features["permutation_importance"].rank(
    ascending=False, method="min"
 )
summary_features["shap_rank"] = summary_features["mean_abs_shap"].rank(
    ascending=False, method="min"
 )
summary_features

,permutation_importance,mean_abs_shap,permutation_rank,shap_rank
sex,0.019808,0.039790,1.0,5.0
capital gains,0.006545,0.014761,2.0,11.0
dividends from stocks,0.004737,0.033563,3.0,7.0
education-num,0.004214,0.065622,4.0,2.0
class of worker_ Private,0.002135,0.002331,5.0,26.0
...,...,...,...,...
class of worker_ Not in universe,-0.015794,0.026544,122.0,9.0
age,-0.018173,0.040505,123.0,4.0
tax filer stat_ Nonfiler,-0.031369,0.037836,124.0,6.0
num persons worked for employer,-0.033250,0.043279,125.0,3.0


In [31]:
new_features = summary_features[(summary_features["permutation_rank"] <= 10) | (summary_features["shap_rank"] <= 10)].index

In [32]:
train_x_new = train_x[new_features]
test_x_new = test_x[new_features]

my_model = RandomForestClassifier(
    n_estimators=500,
    random_state=99,
    min_samples_split=500,
    max_leaf_nodes=50,
).fit(train_x_new, train_y)
test_pred = my_model.predict(test_x_new)

print(classification_report(test_y, test_pred, zero_division=0))
print(confusion_matrix(test_y, test_pred))

              precision    recall  f1-score   support

           0       0.99      0.84      0.91     46753
           1       0.27      0.88      0.41      3128

    accuracy                           0.84     49881
   macro avg       0.63      0.86      0.66     49881
weighted avg       0.95      0.84      0.88     49881

[[39093  7660]
 [  364  2764]]


In [33]:
model_features = list(getattr(my_model, "feature_names_in_", test_x_new.columns))
test_x_aligned = test_x_new.reindex(columns=model_features, fill_value=0)

perm = PermutationImportance(
    my_model,
    random_state=99,
    scoring="f1",
)
perm.fit(test_x_aligned, test_y)

perm_importance = (
    pd.Series(
        perm.feature_importances_,
        index=model_features,
        name="permutation_importance",
    )
    .sort_values(ascending=False)
 )

perm_summary = perm_importance.head(50).to_frame()
perm_summary

,permutation_importance
sex,0.024521
education-num,0.021611
age,0.014274
capital gains,0.010497
dividends from stocks,0.006362
capital losses,0.002066
class of worker_ Private,0.001259
wage per hour,0.000581
veterans benefits,0.000342
hispanic origin_ All other,0.000002


In [34]:
analysis_sample = test_x_aligned.sample(min(1000, len(test_x_aligned)), random_state=99)

explainer = shap.TreeExplainer(my_model)
shap_values_sample = explainer.shap_values(analysis_sample)

if isinstance(shap_values_sample, list):
    local_shap_values = shap_values_sample[1] if len(shap_values_sample) > 1 else shap_values_sample[0]
elif isinstance(shap_values_sample, np.ndarray) and shap_values_sample.ndim == 3:
    local_shap_values = shap_values_sample[:, :, 1]
else:
    local_shap_values = shap_values_sample

shap_importance = (
    pd.Series(
        np.abs(local_shap_values).mean(axis=0),
        index=analysis_sample.columns,
        name="mean_abs_shap",
    )
    .sort_values(ascending=False)
 )

shap_summary = shap_importance.head(50).to_frame()
shap_summary

,mean_abs_shap
weeks worked in year,0.102116
education-num,0.100989
age,0.059711
sex,0.051010
dividends from stocks,0.044593
num persons worked for employer,0.043370
tax filer stat_ Nonfiler,0.041875
detailed household summary in household_ Householder,0.031195
class of worker_ Not in universe,0.022340
marital stat_ Never married,0.016923


In [35]:
summary_features = pd.concat(
    [perm_importance, shap_importance],
    axis=1,
    join="inner",
)

summary_features["permutation_rank"] = summary_features["permutation_importance"].rank(
    ascending=False, method="min"
 )
summary_features["shap_rank"] = summary_features["mean_abs_shap"].rank(
    ascending=False, method="min"
 )
summary_features

,permutation_importance,mean_abs_shap,permutation_rank,shap_rank
sex,0.024521,0.051010,1.0,4.0
education-num,0.021611,0.100989,2.0,2.0
age,0.014274,0.059711,3.0,3.0
capital gains,0.010497,0.016793,4.0,11.0
dividends from stocks,0.006362,0.044593,5.0,5.0
capital losses,0.002066,0.004376,6.0,12.0
class of worker_ Private,0.001259,0.001627,7.0,13.0
wage per hour,0.000581,0.001198,8.0,16.0
veterans benefits,0.000342,0.001590,9.0,14.0
hispanic origin_ All other,0.000002,0.001552,10.0,15.0


In [36]:
final_features = summary_features[(summary_features["shap_rank"] <= 3) | (summary_features["permutation_rank"] <= 3)].index
final_features

Index(['sex', 'education-num', 'age', 'weeks worked in year'], dtype='str')

In [37]:
train_x_new = train_x[final_features]
test_x_new = test_x[final_features]

my_model = RandomForestClassifier(
    n_estimators=500,
    random_state=99,
    min_samples_split=500,
    max_leaf_nodes=50,
).fit(train_x_new, train_y)
test_pred = my_model.predict(test_x_new)

print(classification_report(test_y, test_pred, zero_division=0))
print(confusion_matrix(test_y, test_pred))

              precision    recall  f1-score   support

           0       0.99      0.84      0.91     46753
           1       0.26      0.83      0.40      3128

    accuracy                           0.84     49881
   macro avg       0.62      0.84      0.65     49881
weighted avg       0.94      0.84      0.88     49881

[[39344  7409]
 [  525  2603]]


## Bias

In [38]:
raw_census = load_raw_data(None)

In [39]:
raw_census.groupby("sex")["targets"].value_counts(normalize=True).rename("proportion")

sex     targets 
Female  - 50000.    0.974390
        50000+.     0.025610
Male    - 50000.    0.898272
        50000+.     0.101728
Name: proportion, dtype: float64

In [40]:
group_eval = pd.DataFrame(index=test_y.index)
group_eval["sex"] = test_x_new["sex"].astype(int)
group_eval["y_true"] = test_y
group_eval["y_pred"] = my_model.predict(test_x_new)
group_eval["y_prob"] = my_model.predict_proba(test_x_new)[:, 1]

group_eval.groupby("sex", observed=True)["y_pred"].value_counts(normalize=True)

sex  y_pred
0    0         0.914640
     1         0.085360
1    0         0.673021
     1         0.326979
Name: proportion, dtype: float64

In [41]:
model_features = list(getattr(my_model, "feature_names_in_", test_x.columns))
test_x_aligned = test_x.reindex(columns=model_features, fill_value=0)

analysis_sample = test_x_aligned.sample(min(1000, len(test_x_aligned)), random_state=99)

raw_test = raw_census.loc[test_x_aligned.index].copy()
raw_sample = raw_test.loc[analysis_sample.index].copy()
raw_sample["sex"] = raw_sample["sex"].astype(str).str.strip()

explainer = shap.TreeExplainer(my_model)
shap_values = explainer.shap_values(analysis_sample)

if isinstance(shap_values, list):
    positive_shap = shap_values[1] if len(shap_values) > 1 else shap_values[0]
elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    positive_shap = shap_values[:, :, 1]
else:
    positive_shap = shap_values

shap_frame = pd.DataFrame(
    positive_shap,
    index=analysis_sample.index,
    columns=analysis_sample.columns,
 )

sensitive_feature_importance = pd.Series(
    {
        "sex": shap_frame["sex"].abs().mean(),
    },
    name="mean_abs_shap",
).sort_values(ascending=False)

sensitive_feature_importance.to_frame()

,mean_abs_shap
sex,0.075278


In [44]:
bias_results = raw_sample[["sex"]].copy()
bias_results["predicted_probability"] = my_model.predict_proba(analysis_sample)[:, 1]
bias_results["sex_shap"] = shap_frame["sex"]

sex_shap_summary = bias_results.groupby("sex", observed=True)[
    ["predicted_probability", "sex_shap"]
] .agg(["mean", "median", "std", "count"])

sex_shap_quantiles = bias_results.groupby("sex", observed=True)["sex_shap"].quantile(
    [0.1, 0.5, 0.9]
).unstack()

print("SHAP and probability summary by sex")
display(sex_shap_summary)

SHAP and probability summary by sex


predicted_probability                            sex_shap            \
                        mean    median       std count      mean    median   
sex                                                                          
Female              0.202333  0.124844  0.216929   526 -0.096712 -0.092649   
Male                0.368758  0.267576  0.342913   474  0.051493  0.051957   

                        
             std count  
sex                     
Female  0.046059   526  
Male    0.022277   474

In [45]:
counterfactual_sample = raw_test[["sex", "race"]].copy()
counterfactual_sample["sex_group"] = counterfactual_sample["sex"].astype(str).str.strip()
counterfactual_sample = counterfactual_sample.sample(
    min(250, len(counterfactual_sample)), random_state=7
)

baseline_features = test_x_aligned.loc[counterfactual_sample.index].copy()
baseline_probability = my_model.predict_proba(baseline_features)[:, 1]

sex_groups = counterfactual_sample["sex_group"].value_counts().index.tolist()
sex_swap = {sex_groups[0]: sex_groups[1], sex_groups[1]: sex_groups[0]}
sex_counterfactual = baseline_features.copy()
sex_counterfactual["sex"] = 1 - sex_counterfactual["sex"]
sex_counterfactual_probability = my_model.predict_proba(sex_counterfactual)[:, 1]

counterfactual_results = counterfactual_sample[["sex_group"]].copy()
counterfactual_results["baseline_probability"] = baseline_probability
counterfactual_results["sex_counterfactual_group"] = counterfactual_results["sex_group"].map(sex_swap)
counterfactual_results["sex_counterfactual_probability"] = sex_counterfactual_probability
counterfactual_results["sex_probability_delta"] = (
    sex_counterfactual_probability - baseline_probability
)

counterfactual_effect_summary = pd.DataFrame(
    {
        "mean_abs_delta": [
            counterfactual_results["sex_probability_delta"].abs().mean(),
        ],
        "max_abs_delta": [
            counterfactual_results["sex_probability_delta"].abs().max(),
        ],
        "label_flip_rate": [
            (
                (counterfactual_results["baseline_probability"] >= 0.5)
                != (counterfactual_results["sex_counterfactual_probability"] >= 0.5)
            ).mean(),
        ],
    },
    index=["sex"],
)

display(counterfactual_effect_summary)
counterfactual_results.head()

,mean_abs_delta,max_abs_delta,label_flip_rate
sex,0.119311,0.490771,0.16


,sex_group,baseline_probability,sex_counterfactual_group,sex_counterfactual_probability,sex_probability_delta
149614,Male,0.000012,Female,0.000046,0.000034
26828,Male,0.236334,Female,0.164438,-0.071895
151580,Male,0.000012,Female,0.000046,0.000034
159385,Female,0.011891,Male,0.013361,0.001470
76378,Female,0.000046,Male,0.000012,-0.000034
